# Image Sensing — the CondadosAI unit, runnable

One notebook, one section per lesson, all on the same eighteen raw exposures.
Run the setup cells, then jump to whichever lesson you are reading.

| Section | Lesson |
|---|---|
| 1 | From photons to counts |
| 2 | Sensor noise and dynamic range |
| 3 | The camera response curve |
| 4 | High dynamic range imaging |

**About the images.** The scene is *Luxo Double Checker* from Mark Fairchild's HDR
Photographic Survey. Its terms allow research use and non-commercial publication,
with the source acknowledged; the files are downloaded here and never redistributed.
Fetching one scene pulls about 220 MB out of the survey's raw archive.


## Setup

Versions are pinned so this notebook keeps producing the numbers the articles
publish. Colab ships OpenCV 4.x by default; the pin below replaces it, and Colab
may ask you to restart the runtime afterwards — do that, then carry on from the
cell after this one.


In [ ]:
!pip install -q "opencv-python>=5.0.0,<6.0.0" "numpy>=1.26" "matplotlib>=3.8" "loguru>=0.7" "click>=8.1" "rawpy>=0.21" "exifread>=3.0" "xlrd>=2.0"
!git clone -q https://github.com/CondadosAI/fundamentals-lab.git 2>/dev/null || echo "already cloned"
!pip install -q -e fundamentals-lab

import sys
sys.path.insert(0, "fundamentals-lab/src")


In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt, rawpy
print("opencv", cv2.__version__, "| numpy", np.__version__, "| rawpy", rawpy.__version__)

from click.testing import CliRunner
from fundamentals_lab.cli.sensing_download import sensing_download
print(CliRunner().invoke(sensing_download, []).output[-400:] or "scene ready")


In [ ]:
from fundamentals_lab.sensing import charts, hdrps, noise

paths = sorted((hdrps.scene_dir() / "raw").glob("*.NEF"))
frames = sorted((hdrps.load_raw(p) for p in paths), key=lambda f: f.shutter)
print(len(frames), "exposures from", frames[0].shutter, "s to", frames[-1].shutter, "s")
print(hdrps.scene_metadata())


## 1. From photons to counts

The article's first code block: what the file declares about the sensor. Black
level, the two disagreeing ceilings, and the mosaic layout.


In [ ]:
with rawpy.imread(str(paths[4])) as raw:
    counts = raw.raw_image_visible          # one plane, still a Bayer mosaic
    print(raw.black_level_per_channel)      # [0, 0, 0, 0]
    print(raw.white_level)                  # 4095
    print(raw.camera_white_level_per_channel)  # [3827, 3827, 3827, 3827]


Where the counts actually stop is a measurement, not a tag — and it differs per
colour channel because the analog white balance is applied before the converter.


In [ ]:
longest = frames[-1].counts
for name in ("R", "G1", "G2", "B"):
    plane = noise.channel(longest, name)
    print(f"{name}: max {plane.max()}, clipped fraction {(plane >= plane.max()*0.98).mean():.3f}")


The worked example: the lit chart's white patch across the bracket. The count
doubles with the shutter, then stops.


In [ ]:
white = [p for p in charts.patches("bright") if p.number == 19][0]
ladder = [(f.shutter, float(np.median(white.sample_cfa(f.counts, 0, 1)))) for f in frames]
for shutter, dn in ladder[:7]:
    print(f"{shutter:8.5f} s   {dn:7.1f} DN")

plt.figure(figsize=(5, 3.2))
plt.loglog(*zip(*ladder), "o-")
plt.xlabel("exposure time (s)"); plt.ylabel("count (DN)"); plt.title("one patch, eighteen exposures")
plt.grid(alpha=0.3); plt.show()


## 2. Sensor noise and dynamic range

The article's estimator: variance from the median absolute deviation of frame
differences, over the three frames that share the 30 s exposure.


In [ ]:
def green(frame):
    return noise.channel(frame.counts, "G1").astype(np.float64)

repeats = [f for f in frames if f.shutter == frames[-1].shutter]
a, b, c = (green(f) for f in repeats)

stack = np.stack([a, b, c])
mean = stack.mean(axis=0)

differences = np.concatenate([(stack[0] - stack[1]).ravel(),
                              (stack[0] - stack[2]).ravel(),
                              (stack[1] - stack[2]).ravel()])
sigma_difference = np.median(np.abs(differences)) / 0.6745
variance = sigma_difference ** 2 / 2
print(f"pooled over the whole frame: sigma {np.sqrt(variance):.2f} DN")


Pooled by signal level, that becomes the photon transfer curve. The slope is the
gain in DN per electron; the intercept is where read noise would be, and this data
does not reach far enough down to measure it.


In [ ]:
saturation = noise.clip_level(repeats[-1].counts, "G1")
points = noise.transfer_points([f.counts for f in repeats], "G1", saturation)
fit = noise.fit(points)
print(f"gain {fit.gain_dn_per_electron:.4f} +/- {fit.gain_stderr:.4f} DN/e-  "
      f"({fit.electrons_per_dn:.2f} e-/DN), R2 {fit.r_squared:.4f}")
print(f"intercept {fit.intercept_dn2:.1f} +/- {fit.intercept_stderr:.1f} DN^2")
print(f"full well {fit.full_well(saturation / noise.CLIP_MARGIN):.0f} e-")

xs = [p.mean_dn for p in points]; ys = [p.variance_dn2 for p in points]
plt.figure(figsize=(5, 3.2)); plt.plot(xs, ys, "o")
line = np.linspace(0, max(xs), 50)
plt.plot(line, fit.gain_dn_per_electron * line + fit.intercept_dn2)
plt.xlabel("signal (DN)"); plt.ylabel("variance (DN^2)"); plt.grid(alpha=0.3); plt.show()


Read noise, bounded from a dark corner of the shortest exposure, and the dynamic
range that follows from it.


In [ ]:
dark = noise.dark_noise_bound(frames[0].counts, (100, 100, 900, 900), "G1")
ceiling = saturation / noise.CLIP_MARGIN
print(f"dark corner sigma {dark:.2f} DN = {fit.electrons(dark):.1f} e- (an upper bound)")
print(f"dynamic range >= {fit.dynamic_range_stops(ceiling, dark):.2f} stops")


## 3. The camera response curve

The article's third code block: the same pixels twice, as counts and as code
values.


In [ ]:
import rawpy

with rawpy.imread(str(paths[4])) as raw:
    linear = raw.raw_image_visible.copy()                 # counts, proportional to light
    developed = raw.postprocess(use_camera_wb=True,       # what a viewer would show you
                                output_bps=8, no_auto_bright=True)

grey = cv2.cvtColor(cv2.cvtColor(developed, cv2.COLOR_RGB2BGR), cv2.COLOR_BGR2GRAY)
neutral = [p for p in charts.patches("bright") if p.number in (19, 20, 21, 22, 23, 24)]
for patch in neutral:
    counts = float(np.median(patch.sample_cfa(linear, 0, 1)))
    code = float(np.median(patch.sample(grey)))
    print(f"{patch.key:10s} {counts:7.1f} DN   ->  {code:5.1f} / 255")


In [ ]:
from fundamentals_lab.sensing import response

points = response.response_points(linear, grey, neutral)
gamma = response.fit_gamma(points, ceiling)
print(f"single-exponent fit: gamma {gamma.gamma:.3f} +/- {gamma.stderr:.3f}, R2 {gamma.r_squared:.4f}")

relative = np.array([p.linear_dn / ceiling for p in points])
srgb = response.srgb_encode(relative)
print(f"mean |code - sRGB| = {np.mean(np.abs([p.code_value for p in points] - srgb)):.1f} code values")

grid = np.geomspace(0.002, 1, 200)
plt.figure(figsize=(5, 3.2))
plt.semilogx(grid, response.srgb_encode(grid), label="sRGB")
plt.semilogx(grid, gamma.predict(grid), label=f"gamma {gamma.gamma:.2f}")
plt.semilogx(relative, [p.code_value for p in points], "o", label="measured")
plt.xlabel("light, relative to saturation"); plt.ylabel("code value")
plt.legend(); plt.grid(alpha=0.3); plt.show()


## 4. High dynamic range imaging

The article's fourth code block: the weighted merge, written out. `sensing/hdr.py`
runs the same arithmetic over all four mosaic channels.


In [ ]:
floor_dn, sat = 24.0, saturation
planes = [noise.channel(f.counts, "G1").astype(np.float32) for f in frames]
shutters = [f.shutter for f in frames]

weight_sum = np.zeros_like(planes[0], dtype=np.float64)
value_sum = np.zeros_like(weight_sum)
middle = (floor_dn + sat) / 2

for plane, shutter in zip(planes, shutters):        # one colour channel per plane
    usable = (plane >= floor_dn) & (plane < sat)
    weight = np.where(usable, 1.0 - np.abs(plane - middle) / (middle - floor_dn), 0.0)
    weight_sum += weight
    value_sum += weight * (plane / shutter)

radiance = np.divide(value_sum, weight_sum, out=np.zeros_like(value_sum), where=weight_sum > 0)
print("covered fraction:", float((weight_sum > 0).mean()))


Then the check that makes it a measurement: one scale factor fitted on one patch,
and the other 47 compared against the colorimeter.


In [ ]:
from fundamentals_lab.sensing import hdr
from fundamentals_lab.config import CALIBRATION_ANCHOR, D2X_LUMINANCE_ROW

exposures = [(f.counts, f.shutter) for f in frames]
radiances = {n: hdr.merge(exposures, n, noise.clip_level(frames[-1].counts, n))
             for n in ("R", "G1", "G2", "B")}
with rawpy.imread(str(paths[0])) as raw:
    balance = tuple(float(v) for v in raw.camera_whitebalance[:3])

patches = charts.all_patches()
metered = {p.key: p.luminance for p in hdrps.metered_points()}
luminance = hdr.luminance(radiances, patches, balance, D2X_LUMINANCE_ROW)
scale, predictions = hdr.calibrate(luminance, metered, CALIBRATION_ANCHOR)

errors = np.array([p.error_stops for p in predictions])
print(f"{len(predictions)} points | median |error| {np.median(np.abs(errors)):.3f} stops"
      f" | worst {np.max(np.abs(errors)):.2f}")

plt.figure(figsize=(4.2, 4.2))
plt.loglog([p.metered_cd_m2 for p in predictions], [p.predicted_cd_m2 for p in predictions], "o")
limits = [0.01, 1000]; plt.plot(limits, limits, "--", linewidth=1)
plt.xlabel("metered (cd/m$^2$)"); plt.ylabel("merged (cd/m$^2$)"); plt.grid(alpha=0.3); plt.show()


## Your own camera

Upload a bracket of your own raw files, drop them in `data/hdrps/Yours/raw/`, and
the noise and response sections run on them unchanged. The HDR section needs one
metered luminance, or it reports relative light only.

The numbers the articles publish come from `uv run sensing-experiments`, which
writes `output/sensing_numbers.json` with everything above plus the per-channel
results.
